In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *

STORAGE_ACCOUNT = "dltlearn"
RAW_CONTAINER = "raw"
gold_path = f"abfss://{RAW_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold"

catalog = 'test'
silver_schema = 'silver'
gold_schema = 'gold'

gold_database = f"{catalog}.{gold_schema}"
silver_database = f"{catalog}.{silver_schema}"

df_orders    = spark.read.table (f"{silver_database}.orders")
df_items     = spark.read.table(f"{silver_database}.orderitems")
df_customers = spark.read.table(f"{silver_database}.customer")

In [0]:
df_items.display()

**Order analysis**

In [0]:
df_items_agg = df_items.groupBy("order_id") \
    .agg(
        round(sum("total_item_value"), 2).alias("total_revenue"),
        count("order_item_id").alias("total_items"),
        round(avg("price"), 2).alias("avg_item_price")
    )

df_items_agg.display()    

In [0]:
df_orders.display()

In [0]:
df_customers.display()

In [0]:
df_gold_orders = df_orders \
    .join(df_items_agg, "order_id", "left")\
    .join(df_customers.select("customer_id", "customer_city", "customer_state"),
          "customer_id", "left") \
    .select(
        "order_id",
        "customer_id",
        "customer_city",
        "customer_state",
        "order_status",
        "is_delivered",
        "delivery_delay_days",
        "total_revenue",
        "total_items",
        "avg_item_price",
        "order_purchase_timestamp",
        "order_estimated_delivery_date",
        "order_delivered_customer_date"
    )

df_gold_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{gold_path}/order_analysis")

spark.sql(f"CREATE TABLE IF NOT EXISTS {gold_database}.order_analysis USING DELTA LOCATION '{gold_path}/order_analysis'")

print(f" written {gold_database}.order_analysis ")



In [0]:
df_gold_orders.groupBy('customer_state') \
    .agg(
        round(sum('total_revenue'), 2).alias('total_revenue'),
        count('*').alias('total_orders')
    ) \
    .orderBy('total_revenue', ascending=False) \
    .display()

**Customer Analysis**

In [0]:
# aggregate orders to customer level

df_orders_agg = df_orders.groupBy("customer_id") \
    .agg(
        count("order_id").alias("total_orders"),
        sum(when(col("is_delivered") == True, 1).otherwise(0)).alias("delivered_orders")
    )
# aggregate revenue to customer level
df_customer_revenue = df_items_agg.join(
    df_orders.select("order_id", "customer_id"), "order_id", "left"
).groupBy("customer_id") \
    .agg(
        round(sum("total_revenue"), 2).alias("total_spent"),
        round(avg("total_revenue"), 2).alias("avg_order_value")
    )  


df_gold_customers = df_customers \
    .join(df_orders_agg, "customer_id", "left") \
    .join(df_customer_revenue, "customer_id", "left") \
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_city",
        "customer_state",
        "total_orders",
        "delivered_orders",
        "total_spent",
        "avg_order_value"
    )      

In [0]:
df_gold_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{gold_path}/customer_analysis")

spark.sql(f"CREATE TABLE IF NOT EXISTS {gold_database}.customer_analysis USING DELTA LOCATION '{gold_path}/customer_analysis'")
print(f" written {gold_database}.customer_analysis | number of rows: {df_gold_customers.count()}")